# Greatest Hits — Train on Colab T4

Runs the v1 MLP from cached cochleagrams + visual features. **Does not need the 60 GB raw dataset** — only the ~2 GB cache bundle produced by `python src/prepare_colab_bundle.py` locally.

Before running:
1. Make sure `cache_bundle.tar.gz` is uploaded to Drive at `MyDrive/cv_final_project/cache_bundle.tar.gz` (or edit `BUNDLE_PATH` below).
2. Runtime → Change runtime type → **GPU** (T4 is fine).

In [ ]:
!nvidia-smi -L

In [ ]:
# Clone code from GitHub
!rm -rf /content/cv_final_project
!git clone https://github.com/Owen-M8/cv_final_project.git /content/cv_final_project
%cd /content/cv_final_project

In [ ]:
# Install Python deps. Colab already has torch/torchvision; pycochleagram needs the GitHub install.
!pip install -q --upgrade pip
!pip install -q soundfile librosa opencv-python tqdm
!pip install -q git+https://github.com/mcdermottLab/pycochleagram.git

In [ ]:
# Mount Google Drive and pull the cache bundle
from google.colab import drive
drive.mount('/content/drive')

BUNDLE_PATH = '/content/drive/MyDrive/cv_final_project/cache_bundle.tar.gz'
!ls -lh "$BUNDLE_PATH"

# Extract into project root. The archive contains a top-level cache/ directory.
!tar -xzf "$BUNDLE_PATH" -C /content/cv_final_project/
!ls /content/cv_final_project/cache/ | head -5
!echo "cochleagram files:  $(ls /content/cv_final_project/cache/ | grep _coch.npz | wc -l)"
!echo "feature files:      $(ls /content/cv_final_project/cache/ | grep _feat.npy | wc -l)"
!ls /content/cv_final_project/cache/clip_index.json

In [ ]:
# Sanity check: load the clip index, verify it matches the cache.
import sys
sys.path.insert(0, '/content/cv_final_project/src')
from dataset import load_clip_index
from pathlib import Path

train_clips, test_clips = load_clip_index()
print(f'{len(train_clips)} train clips, {len(test_clips)} test clips')

# Spot-check that the corresponding cache files exist
from visual_features import _feat_cache_path
from dataset import _cache_path
missing_feat = [c for c in train_clips[:200] if not _feat_cache_path(c).exists()]
missing_coch = [c.entry for c in train_clips[:200] if not _cache_path(c.entry.video_id).exists()]
print(f'spot-check (first 200 train clips): {len(missing_feat)} missing features, {len(missing_coch)} missing cochleagrams')

In [ ]:
# Train the v1 MLP. Should run in 5-10 minutes on T4.
%cd /content/cv_final_project
!python src/train.py --epochs 20 --batch-size 64

In [ ]:
# Inference on a held-out clip (first test clip).
# NOTE: inference.py needs to read the video file to extract visual features for new
# clips. To listen to predictions on Colab without uploading 60 GB, we use the cached
# features for an existing test clip and only need cochleagram inversion.
import sys
sys.path.insert(0, '/content/cv_final_project/src')
import numpy as np
import torch
import soundfile as sf
from pathlib import Path

from config import CHECKPOINT_DIR, OUTPUT_DIR, TARGET_SR, ENVELOPE_SR
from cochleagram import cochleagram_to_waveform
from inference import load_checkpoint
from visual_features import _feat_cache_path
from dataset import load_clip_index

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, pca = load_checkpoint(CHECKPOINT_DIR / 'v1.pt', device)

_, test_clips = load_clip_index()
OUT = Path('/content/cv_final_project/outputs/colab_predictions')
OUT.mkdir(parents=True, exist_ok=True)

for i, clip in enumerate(test_clips[:5]):
    feats = np.load(_feat_cache_path(clip))
    z = model(torch.from_numpy(feats).unsqueeze(0).to(device)).detach().cpu().numpy()[0]
    coch_flat = pca.inverse_transform(z[None])[0]
    coch = np.clip(coch_flat.reshape(pca.target_shape), 0.0, None).astype(np.float32)
    wav = cochleagram_to_waveform(coch, sr=TARGET_SR, envelope_sr=ENVELOPE_SR)
    out_path = OUT / f'{i:02d}_{clip.entry.video_id}_t{clip.onset_time:.2f}_{clip.material}.wav'
    sf.write(str(out_path), wav, TARGET_SR)
    print(out_path)

In [ ]:
# Listen to predictions inline
from IPython.display import Audio, display
from pathlib import Path
for p in sorted(Path('/content/cv_final_project/outputs/colab_predictions').glob('*.wav')):
    print(p.name)
    display(Audio(str(p)))

In [ ]:
# Save the trained checkpoint + predictions back to Drive
!mkdir -p /content/drive/MyDrive/cv_final_project/checkpoints
!cp -v checkpoints/v1.pt /content/drive/MyDrive/cv_final_project/checkpoints/
!cp -v checkpoints/v1_history.json /content/drive/MyDrive/cv_final_project/checkpoints/
!cp -rv outputs/colab_predictions /content/drive/MyDrive/cv_final_project/